# BME 574 — How nonlinear fitting works
### Gauss–Newton, why it fails, and the three characters that fix it

In Project 2 you write a nonlinear least-squares fitter, then hand the same problem to `lmfit`
and check the two agree. This notebook explains what the algorithm is doing and why the fix is
what it is.

**Read and run.** Nothing to fill in. About 30 minutes.

---

### Why this is not Project 1

In Project 1 the SVD gave you the answer in one call. There was no iteration, no starting guess,
and no way for the computation to fail. Nonlinear fitting is different in every one of those
respects:

| | Linear (Project 1) | Nonlinear (Project 2) |
|---|---|---|
| Solution | closed form, one shot | iterative, may not converge |
| Starting guess | none needed | **matters enormously** |
| Failure mode | none, really | wrong answer, reported confidently |
| Uncertainty | rarely asked for | the point of the exercise |

### What we will establish

1. Least squares means minimizing a sum of squared residuals — and for a nonlinear model, that
   surface can have valleys, ridges and more than one bottom.
2. **Gauss–Newton** turns the nonlinear problem into a sequence of linear ones. It is about eight
   lines of code.
3. It diverges, spectacularly, from a poor starting guess.
4. **Levenberg–Marquardt** is Gauss–Newton plus a damping term — three characters of code — and
   it is what `lmfit` runs by default.
5. A fitted parameter without an uncertainty is not a result, and the error bar you get for free
   is often wrong.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lmfit

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
BLUE, RED, GREEN = '#2c6fbb', '#c0392b', '#2e6b3e'
print('lmfit', lmfit.__version__)

---
# Part 1 · The problem, and the surface it lives on

We will use a real dataset with a **certified** answer: NIST's *BoxBOD*, biochemical oxygen
demand against incubation time. Six points, two parameters.

$$ y = b_1\,(1 - e^{-b_2 x}) $$

The US National Institute of Standards and Technology publishes this as a test problem for
fitting software, together with the correct parameter values to ten significant figures. So
unlike almost every fit you will ever do, here we know the right answer exactly — which makes it
the right place to find out whether your code works.

In [ ]:
x = np.array([1., 2., 3., 5., 7., 10.])          # incubation time, days
y = np.array([109., 149., 149., 191., 213., 224.])  # oxygen demand, mg/l

# NIST certified values, to 10 significant figures
CERT = np.array([2.1380940889e+02, 5.4723748542e-01])
CERT_SD = np.array([1.2354515176e+01, 1.0455993237e-01])
CERT_RSS = 1.1680088766e+03

def model(p, x):
    b1, b2 = p
    return b1 * (1 - np.exp(-b2 * x))

def residuals(p, x, y):
    return model(p, x) - y

def cost(p, x, y):
    r = residuals(p, x, y)
    return 0.5 * (r ** 2).sum()

print('certified b1 =', CERT[0], ' b2 =', CERT[1])
print('certified RSS =', CERT_RSS)

plt.figure(figsize=(6, 4))
plt.plot(x, y, 'o', ms=8, color=BLUE, label='data')
xs = np.linspace(0, 11, 200)
plt.plot(xs, model(CERT, xs), '-', color=RED, label='certified fit')
plt.xlabel('incubation time (days)'); plt.ylabel('BOD (mg/l)')
plt.title('NIST BoxBOD'); plt.legend(); plt.tight_layout(); plt.show()

## 1.1 · The cost surface

Fitting means finding the parameters that minimize

$$ S(\mathbf{p}) = \tfrac12 \sum_i \left[ f(x_i;\mathbf{p}) - y_i \right]^2 $$

With only two parameters we can draw that surface. Look at the shape of it — this picture
explains everything that follows.

In [ ]:
b1g = np.linspace(50, 400, 260)
b2g = np.linspace(0.02, 2.0, 260)
B1, B2 = np.meshgrid(b1g, b2g)
S = np.array([[cost([a, b], x, y) for a in b1g] for b in b2g])

plt.figure(figsize=(7, 4.8))
cs = plt.contour(B1, B2, np.log10(S), levels=28, cmap='viridis', linewidths=.8)
plt.colorbar(cs, label='log10 cost')
plt.plot(*CERT, '*', ms=20, color=RED, zorder=5, label='certified minimum')
plt.plot(1, 1, 'X', ms=11, color='k', zorder=5, label='NIST start 1 (hard)')
plt.plot(100, 0.75, 'P', ms=11, color=GREEN, zorder=5, label='NIST start 2 (easy)')
plt.xlabel('$b_1$'); plt.ylabel('$b_2$')
plt.title('the cost surface — note the long curved valley')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

print('The minimum sits at the end of a narrow, curved valley.')
print('NIST ships TWO official starting points: one inside the valley, one outside it.')

---
# Part 2 · Gauss–Newton

## 2.1 · The idea

The model is nonlinear in the parameters, so we cannot solve for the minimum directly. But near
any point $\mathbf{p}$, the model is *approximately* linear:

$$ f(x;\mathbf{p}+\delta) \approx f(x;\mathbf{p}) + \mathbf{J}\,\delta $$

where $\mathbf{J}$ is the **Jacobian** — the matrix of derivatives of each prediction with
respect to each parameter, one row per data point and one column per parameter.

Substituting that approximation into the least-squares problem gives a *linear* least-squares
problem for the step $\delta$, whose solution is the normal equations:

$$ (\mathbf{J}^{\!\top}\mathbf{J})\,\delta = -\mathbf{J}^{\!\top}\mathbf{r} $$

So: linearize, solve, step, repeat. That is the whole algorithm.

In [ ]:
def jacobian(p, x):
    """d(prediction)/d(parameter). One row per point, one column per parameter."""
    b1, b2 = p
    e = np.exp(-b2 * x)
    return np.column_stack([1 - e,          # d/db1
                            b1 * x * e])    # d/db2


def jacobian_fd(p, x, eps=1e-6):
    """The same thing by finite differences — our independent check."""
    p = np.asarray(p, float)
    cols = []
    for j in range(len(p)):
        step = np.zeros_like(p); step[j] = eps * max(abs(p[j]), 1.0)
        cols.append((model(p + step, x) - model(p - step, x)) / (2 * step[j]))
    return np.column_stack(cols)


p_test = np.array([150.0, 0.4])
Ja, Jf = jacobian(p_test, x), jacobian_fd(p_test, x)
print('analytic Jacobian:\n', Ja)
print('\nmax abs difference vs finite differences:', np.abs(Ja - Jf).max())
assert np.abs(Ja - Jf).max() < 1e-6
print('\nJacobian check PASSED — this is the first thing to verify, always.')

> **Why check the Jacobian first?** A wrong derivative does not raise an error. It produces a
> fitter that converges slowly, or to the wrong place, or not at all — and you will blame the
> algorithm. Two lines of finite differences rules it out in advance. This is the single highest
> value check in the whole project.

## 2.2 · Eight lines

Here it is. `np.linalg.lstsq` solves the linear sub-problem; we record the path so we can watch
what happens.

In [ ]:
def gauss_newton(p0, x, y, n_iter=60):
    p = np.array(p0, float)
    path = [p.copy()]
    for _ in range(n_iter):
        r = residuals(p, x, y)
        J = jacobian(p, x)
        if not (np.all(np.isfinite(r)) and np.all(np.isfinite(J))):
            break            # it has already left the building; stop cleanly
        try:
            step = np.linalg.lstsq(J, -r, rcond=None)[0]
        except np.linalg.LinAlgError:
            break
        p = p + step
        path.append(p.copy())
        if not np.all(np.isfinite(p)):
            break
        if np.linalg.norm(step) < 1e-12 * (1 + np.linalg.norm(p)):
            break
    return p, np.array(path)


p_easy, path_easy = gauss_newton([100.0, 0.75], x, y)
print('from NIST start 2 (100, 0.75)')
print('  result   ', p_easy)
print('  certified', CERT)
print('  rel error', np.abs(p_easy - CERT) / CERT)
print('  iterations', len(path_easy) - 1)

From a good starting point, Gauss–Newton nails the certified answer. Now the other official
starting point.

In [ ]:
with np.errstate(over='ignore', invalid='ignore'):
    p_hard, path_hard = gauss_newton([1.0, 1.0], x, y)

print('from NIST start 1 (1, 1)')
print('  result   ', p_hard)
print('  certified', CERT)
with np.errstate(over='ignore', invalid='ignore'):
    c_hard = cost(p_hard, x, y)
print('  cost at result   ', c_hard)
print('  cost at certified', cost(CERT, x, y))
print()
print('It did not just stop early. Look at where it went:')
with np.errstate(over='ignore', invalid='ignore'):
    for i, q in enumerate(path_hard[:8]):
        print(f'  iter {i}:  b1 = {q[0]:14.4f}   b2 = {q[1]:14.4f}   '
              f'cost = {cost(q, x, y):14.4f}')
print()
print('One step. b2 went from 1 to about -92, the exponential overflowed,')
print('and the cost became infinite. There is no recovering from that.')

## 2.3 · Why it blows up

Gauss–Newton trusts its linear approximation completely. It computes the step that would be
optimal *if the model were linear*, and takes all of it. When the starting point is far from the
minimum — or the valley is curved, as ours is — that step can be enormous and point somewhere
useless. The next iteration starts from an even worse place, and the process runs away.

Let us watch both paths on the cost surface.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
for ax, (path, ttl, ok) in zip(axes, [
        (path_easy, 'start 2 (100, 0.75) — converges', True),
        (path_hard, 'start 1 (1, 1) — diverges', False)]):
    cs = ax.contour(B1, B2, np.log10(S), levels=28, cmap='viridis', linewidths=.7)
    vis = path[np.all(np.isfinite(path), axis=1)]
    vis = vis[(np.abs(vis[:, 0]) < 1e6) & (np.abs(vis[:, 1]) < 1e6)]
    ax.plot(vis[:, 0], vis[:, 1], 'o-', color=GREEN if ok else RED, ms=4, lw=1.4)
    ax.plot(*CERT, '*', ms=18, color=RED, zorder=5)
    ax.set_xlim(b1g[0], b1g[-1]); ax.set_ylim(b2g[0], b2g[-1])
    ax.set_xlabel('$b_1$'); ax.set_ylabel('$b_2$'); ax.set_title(ttl, fontsize=11)
fig.suptitle('Gauss–Newton, same code, two starting points', y=1.02)
fig.tight_layout(); plt.show()

print('Right-hand panel: the very first step leaves the plot entirely.')

---
# Part 3 · Levenberg–Marquardt — the fix

The problem is that Gauss–Newton takes the full linearized step even when it has no business
trusting it. Levenberg's idea: add a penalty on the *size* of the step.

$$ (\mathbf{J}^{\!\top}\mathbf{J} + \lambda \mathbf{I})\,\delta = -\mathbf{J}^{\!\top}\mathbf{r} $$

That is the entire difference: `J.T @ J` becomes `J.T @ J + lam * I`.

- When $\lambda \to 0$ you recover Gauss–Newton — fast, and right near the minimum.
- When $\lambda$ is large the step shrinks and turns towards the downhill gradient direction —
  slow, but safe.

And $\lambda$ is adapted as you go: **if a step reduced the cost, accept it and trust more
(decrease $\lambda$); if it made things worse, reject it and trust less (increase $\lambda$).**
That adaptive trust is what makes the method robust.

In [ ]:
def levenberg_marquardt(p0, x, y, n_iter=200, lam=1e-2):
    p = np.array(p0, float)
    path, lams = [p.copy()], [lam]
    c = cost(p, x, y)
    for _ in range(n_iter):
        r = residuals(p, x, y)
        J = jacobian(p, x)
        A = J.T @ J + lam * np.eye(len(p))     # <-- the whole difference
        g = -J.T @ r
        try:
            step = np.linalg.solve(A, g)
        except np.linalg.LinAlgError:
            lam *= 10; continue
        p_new = p + step
        c_new = cost(p_new, x, y)
        if np.isfinite(c_new) and c_new < c:
            p, c = p_new, c_new                # accept, and trust more
            lam = max(lam * 0.3, 1e-12)
            path.append(p.copy()); lams.append(lam)
            if np.linalg.norm(step) < 1e-12 * (1 + np.linalg.norm(p)):
                break
        else:
            lam *= 10                          # reject, and trust less
            lams.append(lam)
            if lam > 1e12:
                break
    return p, np.array(path), np.array(lams)


p_lm, path_lm, lam_lm = levenberg_marquardt([1.0, 1.0], x, y)
print('Levenberg-Marquardt from the SAME hard start (1, 1)')
print('  result    ', p_lm)
print('  certified ', CERT)
print('  rel error ', np.abs(p_lm - CERT) / CERT)
print('  RSS       ', 2 * cost(p_lm, x, y), ' certified', CERT_RSS)
print('  accepted steps', len(path_lm) - 1)
assert np.abs(p_lm - CERT).max() / CERT.max() < 1e-5
print()
print('Same starting point. Same Jacobian. Three characters of difference.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
cs = axes[0].contour(B1, B2, np.log10(S), levels=28, cmap='viridis', linewidths=.7)
axes[0].plot(path_lm[:, 0], path_lm[:, 1], 'o-', color=GREEN, ms=4, lw=1.5,
             label='LM (converges)')
vis = path_hard[np.all(np.isfinite(path_hard), axis=1)][:3]
axes[0].plot(vis[:, 0], vis[:, 1], 'o-', color=RED, ms=4, lw=1.5, label='GN (leaves)')
axes[0].plot(*CERT, '*', ms=18, color=RED, zorder=5)
axes[0].set_xlim(b1g[0], b1g[-1]); axes[0].set_ylim(b2g[0], b2g[-1])
axes[0].set_xlabel('$b_1$'); axes[0].set_ylabel('$b_2$')
axes[0].set_title('same start, two algorithms'); axes[0].legend(fontsize=8)

axes[1].semilogy(lam_lm, '.-', lw=1.2)
axes[1].set_xlabel('LM iteration'); axes[1].set_ylabel('damping $\\lambda$')
axes[1].set_title('$\\lambda$ rises when a step fails, falls when it works')
fig.tight_layout(); plt.show()

---
# Part 4 · Now use a real package — `lmfit`

You have written the algorithm, so you know what it does. From here on, use a library: it handles
scaling, convergence tests, bounds and edge cases that took years to get right.

We use **`lmfit`** rather than raw `scipy.optimize`. It is built on the same MINPACK
Levenberg–Marquardt routine, so the numbers agree, but it gives you **named parameters with
bounds**, a fit report that includes **parameter correlations**, and proper **confidence
intervals** — all of which you need for Project 2 and none of which `curve_fit` gives you
directly.

In [ ]:
def boxbod(x, b1, b2):
    return b1 * (1 - np.exp(-b2 * x))

mod = lmfit.Model(boxbod)
pars = mod.make_params(b1=100, b2=0.75)      # named, not positional
result = mod.fit(y, pars, x=x)

print(result.fit_report())

Read that report carefully — there is a lot in it.

- **`chi-square`** is the residual sum of squares. Compare it to the certified value.
- **`[[Correlations]]`** at the bottom: $b_1$ and $b_2$ are correlated at $-0.73$. The two
  parameters are not independently determined — if one goes up, the other can come down and
  partly compensate. That is the beginning of an identifiability problem.
- The `+/-` values are standard errors from the covariance matrix.

In [ ]:
print('OUR LM vs lmfit vs NIST certified\n')
print(f"{'':12s} {'b1':>16s} {'b2':>16s}")
print(f"{'our LM':12s} {p_lm[0]:16.8f} {p_lm[1]:16.8f}")
print(f"{'lmfit':12s} {result.params['b1'].value:16.8f} {result.params['b2'].value:16.8f}")
print(f"{'certified':12s} {CERT[0]:16.8f} {CERT[1]:16.8f}")
print()
print(f"{'our RSS':12s} {2 * cost(p_lm, x, y):16.8f}")
print(f"{'lmfit chisq':12s} {result.chisqr:16.8f}")
print(f"{'certified':12s} {CERT_RSS:16.8f}")
print()
agree = np.abs(p_lm - np.array([result.params['b1'].value, result.params['b2'].value]))
print('our LM agrees with lmfit to', agree.max())
print()
print('Look closely: our LM matches the certified b1 to 8 digits, lmfit to 5.')
print('That is not a bug in lmfit - it is a looser default convergence tolerance,')
print('which is the right trade for everyday use. Tighten it with fit_kws if you')
print('ever need more, pass tighter tolerances through fit_kws:')
print('    mod.fit(y, pars, x=x, fit_kws=dict(ftol=1e-14, xtol=1e-14))')
print()
print('Both agree with NIST to well within the uncertainty. Three routes, one answer.')

## 4.1 · Standard errors, by hand and from the package

The covariance of the fitted parameters is estimated from the Jacobian at the solution:

$$ \mathrm{Cov} \approx s^2 (\mathbf{J}^{\!\top}\mathbf{J})^{-1}, \qquad
   s^2 = \frac{\mathrm{RSS}}{n - p} $$

and the standard errors are the square roots of its diagonal. This is worth computing yourself
once, so the number in the report stops being magic.

In [ ]:
J = jacobian(p_lm, x)
n, npar = len(y), len(p_lm)
rss = 2 * cost(p_lm, x, y)
s2 = rss / (n - npar)
cov = s2 * np.linalg.inv(J.T @ J)
se = np.sqrt(np.diag(cov))

print(f"{'':12s} {'se(b1)':>14s} {'se(b2)':>14s}")
print(f"{'by hand':12s} {se[0]:14.8f} {se[1]:14.8f}")
print(f"{'lmfit':12s} {result.params['b1'].stderr:14.8f} {result.params['b2'].stderr:14.8f}")
print(f"{'certified':12s} {CERT_SD[0]:14.8f} {CERT_SD[1]:14.8f}")
print()
corr = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
print(f'correlation from our covariance matrix: {corr:.4f}')
print(f"correlation lmfit reports            : {result.params['b1'].correl['b2']:.4f}")

---
# Part 5 · The error bar you get for free is often wrong

Those standard errors assume the cost surface is a nice quadratic bowl around the minimum — that
the parameters are, locally, jointly Gaussian. Look again at the contour plot from Part 1. That
valley is *curved*. The assumption is wrong here, and it is wrong in most nonlinear models.

`lmfit` can compute proper confidence intervals by **profiling**: for each parameter, walk it away
from its best value, re-optimizing everything else, until the fit degrades by a statistically
meaningful amount. Slower, and honest.

In [ ]:
ci = lmfit.conf_interval(result.minimizer if hasattr(result, 'minimizer') else result,
                         result, sigmas=[1, 2])
print(lmfit.ci_report(ci, ndigits=4))

In [ ]:
best = {k: result.params[k].value for k in ('b1', 'b2')}
print('symmetric (+/- stderr) vs profiled confidence intervals, 1 sigma\n')
for k in ('b1', 'b2'):
    s = result.params[k].stderr
    lo = ci[k][1][1]; hi = ci[k][3][1]    # idx 1 = -1 sigma, idx 3 = +1 sigma
    print(f'{k}:  best {best[k]:10.4f}')
    print(f'      symmetric   [{best[k] - s:10.4f}, {best[k] + s:10.4f}]   (+/- {s:.4f})')
    print(f'      profiled    [{lo:10.4f}, {hi:10.4f}]   (-{best[k] - lo:.4f} / +{hi - best[k]:.4f})')
    print()
print('The profiled intervals are ASYMMETRIC: the upper arm is longer than the lower.')
print('The symmetric error bar is an approximation, and on a curved valley it')
print('misstates the uncertainty. Which direction it errs in depends on the model.')

> **The rule for Project 2.** Report a parameter with an uncertainty, and say where the
> uncertainty came from. If the parameters are strongly correlated or the intervals are
> noticeably asymmetric, say that too — it means the data do not pin the parameters down
> independently, which is a scientific finding about your experiment, not a numerical nuisance.

---
# Part 6 · Two answers, one fit

The worst case is not a fit that fails. It is a fit that succeeds — twice, differently.

Here is a real example you will meet in the pharmacokinetics tracks. The one-compartment oral
absorption model is

$$ C(t) = \frac{D\,k_a}{V(k_a - k_e)}\left(e^{-k_e t} - e^{-k_a t}\right) $$

Absorption rate $k_a$ and elimination rate $k_e$ enter almost symmetrically. Swap them, adjust
$V$, and you get **the same curve**. This is called *flip-flop kinetics*, and it is not a
numerical artefact — the data genuinely cannot tell the two apart.

In [ ]:
# one real subject from the Theophylline dataset (R datasets::Theoph, subject 5)
t5 = np.array([0, .3, .52, 1, 2.02, 3.5, 5.02, 7.02, 9.1, 12, 24.35])
c5 = np.array([0, 2.02, 5.63, 11.4, 9.33, 8.74, 7.56, 7.09, 5.9, 4.37, 1.57])
D5 = 5.86

def oral(t, ka, ke, V, D=D5):
    d = ka - ke
    d = np.where(np.abs(d) < 1e-12, 1e-12, d)
    return (D * ka / (V * d)) * (np.exp(-ke * t) - np.exp(-ka * t))

omod = lmfit.Model(oral, independent_vars=['t'])
fits = {}
for label, (ka0, ke0) in [('start ka > ke', (1.5, 0.08)),
                          ('start ka < ke', (0.08, 1.5))]:
    p = omod.make_params(ka=dict(value=ka0, min=1e-4),
                         ke=dict(value=ke0, min=1e-4),
                         V=dict(value=0.5, min=1e-3))
    fits[label] = omod.fit(c5, p, t=t5)

print(f"{'':16s} {'ka':>10s} {'ke':>10s} {'V':>10s} {'chi-square':>14s}")
for label, r in fits.items():
    v = r.params
    print(f"{label:16s} {v['ka'].value:10.4f} {v['ke'].value:10.4f} "
          f"{v['V'].value:10.4f} {r.chisqr:14.8f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
ts = np.linspace(0, 25, 400)
for a, (label, r) in zip(ax, fits.items()):
    a.plot(t5, c5, 'o', ms=7, color=BLUE, label='subject 5')
    a.plot(ts, r.eval(t=ts), '-', color=RED, lw=1.8, label='fit')
    v = r.params
    a.set_title(f"{label}\nka={v['ka'].value:.3f}  ke={v['ke'].value:.3f}  "
                f"V={v['V'].value:.3f}", fontsize=10)
    a.set_xlabel('time (h)'); a.set_ylabel('conc (mg/l)'); a.legend(fontsize=8)
fig.suptitle('Two different parameter sets. The same curve.', y=1.04)
fig.tight_layout(); plt.show()

vals = [(r.params['ka'].value, r.params['ke'].value, r.params['V'].value)
        for r in fits.values()]
print(f'volume of distribution differs by a factor of {max(v[2] for v in vals) / min(v[2] for v in vals):.1f}x')
print('between the two solutions — and the fit quality is identical.')

> **This is why the project asks for more than a number.** Both fits converged. Both have small
> residuals. Both would pass any check based on goodness of fit alone. Only a multi-start, or
> prior physiological knowledge about which rate should be faster, distinguishes them — and a
> clinician reading *V* off the wrong one is out by an order of magnitude.

The defence is cheap: **start the fit from several places and see whether you always land in the
same spot.** That is one loop.

---
# What to take to the project

### The algorithm

```python
r = residuals(p, x, y)          # what the model gets wrong
J = jacobian(p, x)              # how each prediction moves with each parameter
A = J.T @ J + lam * np.eye(len(p))   # <- LM's damping; drop it and you have Gauss-Newton
step = np.linalg.solve(A, -J.T @ r)
p = p + step                    # if the cost fell: accept, lower lam. If not: reject, raise lam.
```

### The seven things worth remembering

| | |
|---|---|
| **1** | **Check your Jacobian against finite differences before anything else.** A wrong derivative fails silently. |
| **2** | Gauss–Newton trusts its linear approximation completely, and diverges when that trust is misplaced. |
| **3** | Levenberg–Marquardt damps the step and adapts how much it trusts. That is the only difference. |
| **4** | The starting guess is part of the method, not a detail. NIST ships two on purpose. |
| **5** | Standard errors come from $s^2(\mathbf{J}^{\!\top}\mathbf{J})^{-1}$ and assume a quadratic bowl. |
| **6** | Profiled confidence intervals are asymmetric when that assumption fails — which is often. |
| **7** | **Converging is not the same as being right.** Start from several places and check. |

### One practical `lmfit` note

Inside a residual function written for `lmfit.minimize`, the `Parameters` object holds
`Parameter` objects, not numbers. Use `p['ka'].value`, or `p.valuesdict()` to get them all at
once. Passing the `Parameter` straight into NumPy raises a confusing `__array__` error.